KNN

In [ ]:

# ============================================================
# KNN (K-NEAREST NEIGHBORS) - HEART DISEASE CLASSIFICATION
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("heart.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

# ============================================================
# 2. BASIC EXPLORATION
# ============================================================

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["target"].value_counts())

# ============================================================
# 3. TARGET VARIABLE
# ============================================================

# 0 = No Heart Disease
# 1 = Heart Disease

X = df.drop(columns=["target"])
y = df["target"]

# ============================================================
# 4. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

# ============================================================
# 5. CREATE KNN PIPELINE
# ============================================================

model_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(
        n_neighbors=5,
        metric="minkowski",
        p=2
    ))
])

# ============================================================
# 6. TRAIN THE MODEL
# ============================================================

model_knn.fit(X_train, y_train)

# ============================================================
# 7. PREDICTIONS
# ============================================================

y_pred_knn = model_knn.predict(X_test)
y_proba_knn = model_knn.predict_proba(X_test)[:, 1]

# ============================================================
# 8. MODEL EVALUATION
# ============================================================

print("\nKNN MODEL RESULTS")

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred_knn))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_knn,
    target_names=["No Disease", "Disease"],
    zero_division=0
))

print("\nROC-AUC:")
print(roc_auc_score(y_test, y_proba_knn))

# ============================================================
# 9. PREDICTION ON A NEW PATIENT
# ============================================================

new_patient = pd.DataFrame([{
    "age": 58,
    "sex": 1,
    "cp": 2,
    "trestbps": 130,
    "chol": 246,
    "fbs": 0,
    "restecg": 1,
    "thalach": 152,
    "exang": 0,
    "oldpeak": 1.0,
    "slope": 2,
    "ca": 0,
    "thal": 2
}])

# Arrange columns in the same order as training data
new_patient = new_patient[X.columns]

pred_class = model_knn.predict(new_patient)[0]
pred_proba = model_knn.predict_proba(new_patient)[0, 1]

label = "Heart Disease" if pred_class == 1 else "No Heart Disease"

print("\nNEW PATIENT PREDICTION")
print("Predicted Class:", pred_class, f"({label})")
print("Predicted Probability:", round(pred_proba, 4))
